In [3]:
from flask import (
    Flask,
    request,
    redirect,
    url_for,
    session,
    render_template_string,
    flash
)

import random
import re
import ssl
import smtplib
import secrets
import time

from email.message import EmailMessage
from cryptography.fernet import Fernet
import mysql.connector

from threading import Thread
from IPython.display import display, HTML



#CONFIGURATION


app = Flask(__name__)


app.secret_key = secrets.token_hex(32)

PORTFOLIO_URL = (
    "https://khs-portfolio-60090942218."
    "development.catalystserverless.in/app/index.html"
)

# MYSQL CONFIGURATION
MYSQL_HOST = "127.0.0.1"
MYSQL_PORT = 3306
MYSQL_USER = "root"
MYSQL_PASSWORD = "Hari_2004"
MYSQL_DATABASE = "otp_auth"

# EMAIL CONFIGURATION
EMAIL_SENDER = "hamzaalimizari98@gmail.com"
EMAIL_PASSWORD = "tevt aklx bhlm zvxo"


pending_registrations = {}


otp_attempts = {}



#MYSQL CONNECTION


def get_db_connection():

    return mysql.connector.connect(
        host=MYSQL_HOST,
        port=MYSQL_PORT,
        user=MYSQL_USER,
        password=MYSQL_PASSWORD,
        database=MYSQL_DATABASE
    )



# EMAIL OTP FUNCTION


def send_otp(username, otp):

    em = EmailMessage()

    em["From"] = EMAIL_SENDER
    em["To"] = username
    em["Subject"] = "OTP Verification"

    em.set_content(
        f"Your OTP is: {otp}\n\n"
        "This OTP expires in 5 minutes.\n"
        "Please do not share it with anyone."
    )

    context = ssl.create_default_context()

    with smtplib.SMTP_SSL(
        "smtp.gmail.com",
        465,
        context=context,
        timeout=20
    ) as smtp:

        smtp.login(EMAIL_SENDER, EMAIL_PASSWORD)
        smtp.send_message(em)


# PASSWORD VALIDATION


def validate_password(password):

    if len(password) < 6:
        return "Password must contain at least 6 characters."

    if not re.search(r"[A-Z]", password):
        return "Password must contain an uppercase letter."

    if not re.search(r"[0-9]", password):
        return "Password must contain a number."

    if not re.search(r"[!@#$%^&*()_=+\-/]", password):
        return "Password must contain a special character."

    return None



# 5. COMMON HTML TEMPLATE


BASE_STYLE = """
<style>
    * {
        box-sizing: border-box;
    }

    body {
        font-family: Arial, sans-serif;
        background: #eff6ff;
        margin: 0;
        padding: 30px 15px;
        min-height: 100vh;
        display: flex;
        justify-content: center;
        align-items: center;
    }

    .container {
        width: 100%;
        max-width: 400px;
        background: white;
        padding: 35px;
        border-radius: 14px;
        box-shadow: 0 5px 25px rgba(0,0,0,0.10);
    }

    h1, h2 {
        color: #1d4ed8;
        text-align: center;
    }

    p {
        color: #475569;
        text-align: center;
        line-height: 1.6;
    }

    label {
        display: block;
        margin-top: 15px;
        font-weight: bold;
        color: #334155;
    }

    input {
        width: 100%;
        padding: 12px;
        margin-top: 7px;
        border: 1px solid #cbd5e1;
        border-radius: 6px;
        font-size: 15px;
    }

    button, .button {
        display: block;
        width: 100%;
        padding: 13px;
        margin-top: 20px;
        background: #2563eb;
        color: white;
        border: none;
        border-radius: 6px;
        font-size: 16px;
        text-decoration: none;
        text-align: center;
        cursor: pointer;
    }

    button:hover, .button:hover {
        background: #1d4ed8;
    }

    .green {
        background: #0f766e;
    }

    .green:hover {
        background: #115e59;
    }

    .message {
        padding: 10px;
        background: #eff6ff;
        color: #1e40af;
        border-radius: 6px;
        margin-bottom: 10px;
    }

    .back {
        display: block;
        text-align: center;
        margin-top: 20px;
        color: #2563eb;
    }
</style>
"""


def render_page(title, content):

    template = """
    <!DOCTYPE html>
    <html lang="en">

    <head>
        <meta charset="UTF-8">
        <meta name="viewport"
              content="width=device-width, initial-scale=1.0">

        <title>{{ title }}</title>

        {{ style|safe }}
    </head>

    <body>

        <div class="container">

            {% with messages = get_flashed_messages() %}
                {% for message in messages %}
                    <div class="message">{{ message }}</div>
                {% endfor %}
            {% endwith %}

            {{ content|safe }}

        </div>

    </body>
    </html>
    """

    return render_template_string(
        template,
        title=title,
        content=content,
        style=BASE_STYLE
    )


# ============================================================
# 6. PAGE 1: HOME
# ============================================================

@app.route("/")
def home():

    content = """
    <h1>Welcome!</h1>

    <p>Email OTP Authentication System</p>

    <p>
        Securely create your account or log in
        to access your portfolio.
    </p>

    <a class="button" href="/login">Login</a>

    <a class="button green" href="/signup">Sign Up</a>
    """

    return render_page("Home", content)


# ============================================================
# 7. PAGE 2: SIGN UP
# ============================================================

@app.route("/signup", methods=["GET", "POST"])
def signup():

    if request.method == "GET":

        content = """
        <h2>Create Account</h2>

        <form method="POST" action="/signup">

            <label>Email Address</label>

            <input
                type="email"
                name="email"
                maxlength="254"
                required
            >

            <label>Create Password</label>

            <input
                type="password"
                name="password"
                minlength="6"
                required
            >

            <label>Confirm Password</label>

            <input
                type="password"
                name="confirm_password"
                minlength="6"
                required
            >

            <button type="submit">Send OTP</button>

        </form>

        <a class="back" href="/">Back to Home</a>
        """

        return render_page("Sign Up", content)

    # GET FORM DATA

    username = request.form.get("email", "").strip().lower()
    password = request.form.get("password", "")
    confirm_password = request.form.get("confirm_password", "")

    # EMAIL VALIDATION

    email_pattern = (
        r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"
    )

    if not re.fullmatch(email_pattern, username):

        flash("Please enter a valid email address.")
        return redirect(url_for("signup"))

    # PASSWORD VALIDATION

    error = validate_password(password)

    if error:

        flash(error)
        return redirect(url_for("signup"))

    if password != confirm_password:

        flash("Passwords do not match.")
        return redirect(url_for("signup"))

    # CHECK EXISTING USER

    db = None
    cursor = None

    try:

        db = get_db_connection()
        cursor = db.cursor()

        cursor.execute(
            "SELECT id FROM users WHERE email = %s",
            (username,)
        )

        existing_user = cursor.fetchone()

        if existing_user:

            flash("Account already exists. Please log in.")
            return redirect(url_for("login"))

    except mysql.connector.Error:

        app.logger.exception("Database error during signup")

        flash("Unable to connect to the database.")
        return redirect(url_for("signup"))

    finally:

        if cursor is not None:
            cursor.close()

        if db is not None and db.is_connected():
            db.close()

    # GENERATE OTP

    otp = str(random.randint(1000, 9999))

    try:

        send_otp(username, otp)

    except Exception:

        app.logger.exception("OTP email delivery failed")

        flash("Unable to send OTP. Check your email configuration.")
        return redirect(url_for("signup"))

    # STORE TEMPORARY DATA SERVER-SIDE

    registration_id = secrets.token_urlsafe(32)

    pending_registrations[registration_id] = {
        "email": username,
        "password": password,
        "otp": otp,
        "created_at": time.time(),
        "attempts": 0
    }

    session["registration_id"] = registration_id

    return redirect(url_for("verify_otp"))



# OTP VERIFICATION


@app.route("/verify-otp", methods=["GET", "POST"])
def verify_otp():

    registration_id = session.get("registration_id")

    registration = pending_registrations.get(registration_id)

    if registration is None:

        flash("Please start the registration process again.")
        return redirect(url_for("signup"))

    # OTP EXPIRATION: 5 MINUTES

    if time.time() - registration["created_at"] > 300:

        pending_registrations.pop(registration_id, None)
        session.pop("registration_id", None)

        flash("OTP expired. Please sign up again.")
        return redirect(url_for("signup"))

    if request.method == "GET":

        content = """
        <h2>Email Verification</h2>

        <p>Enter the 4-digit OTP sent to your email.</p>

        <form method="POST" action="/verify-otp">

            <label>Enter OTP</label>

            <input
                type="text"
                name="otp"
                inputmode="numeric"
                pattern="[0-9]{4}"
                maxlength="4"
                required
            >

            <button type="submit">Verify OTP</button>

        </form>

        <a class="back" href="/signup">Cancel Registration</a>
        """

        return render_page("Verify OTP", content)

    entered_otp = request.form.get("otp", "").strip()

    registration["attempts"] += 1

    if registration["attempts"] > 3:

        pending_registrations.pop(registration_id, None)
        session.pop("registration_id", None)

        flash("Maximum attempts reached. Please register again.")
        return redirect(url_for("signup"))

    if not re.fullmatch(r"[0-9]{4}", entered_otp):

        flash("Enter a valid 4-digit OTP.")
        return redirect(url_for("verify_otp"))

    if not secrets.compare_digest(entered_otp, registration["otp"]):

        flash("Incorrect OTP. Try again.")
        return redirect(url_for("verify_otp"))

 
    # ENCRYPT PASSWORD USING FERNET


    username = registration["email"]
    password = registration["password"]

    key = Fernet.generate_key()
    cipher = Fernet(key)

    encrypted_password = cipher.encrypt(
        password.encode()
    ).decode()

    secret_key = key.decode()


    # SAVE ACCOUNT TO MYSQL


    db = None
    cursor = None

    try:

        db = get_db_connection()
        cursor = db.cursor()

        cursor.execute(
            """
            INSERT INTO users
            (email, encrypted_password, secret_key)
            VALUES (%s, %s, %s)
            """,
            (username, encrypted_password, secret_key)
        )

        db.commit()

    except mysql.connector.Error as error:

        if db is not None and db.is_connected():
            db.rollback()

        app.logger.exception("Account creation failed")

        # Duplicate account or another database issue
        if getattr(error, "errno", None) == 1062:

            flash("This email is already registered.")

        else:

            flash("Unable to create the account. Please try again.")

        return redirect(url_for("signup"))

    finally:

        if cursor is not None:
            cursor.close()

        if db is not None and db.is_connected():
            db.close()



    pending_registrations.pop(registration_id, None)
    session.pop("registration_id", None)

    flash("Registration successful! Please log in.")

    return redirect(url_for("login"))



# LOGIN


@app.route("/login", methods=["GET", "POST"])
def login():

    if request.method == "GET":

        content = """
        <h2>Login</h2>

        <form method="POST" action="/login">

            <label>Email Address</label>

            <input
                type="email"
                name="email"
                required
            >

            <label>Password</label>

            <input
                type="password"
                name="password"
                required
            >

            <button type="submit">Login</button>

        </form>

        <p>Don't have an account?</p>

        <a class="button green" href="/signup">Sign Up</a>

        <a class="back" href="/">Back to Home</a>
        """

        return render_page("Login", content)

    username = request.form.get("email", "").strip().lower()
    password = request.form.get("password", "")

    db = None
    cursor = None

    try:

        db = get_db_connection()
        cursor = db.cursor()

        cursor.execute(
            """
            SELECT encrypted_password, secret_key
            FROM users
            WHERE email = %s
            """,
            (username,)
        )

        user = cursor.fetchone()

        if user is None:

            flash("Invalid email or password.")
            return redirect(url_for("login"))

        encrypted_password = user[0]
        secret_key = user[1]

        if not encrypted_password or not secret_key:

            flash("Account credentials are incomplete.")
            return redirect(url_for("login"))

        cipher = Fernet(secret_key.encode())

        decrypted_password = cipher.decrypt(
            encrypted_password.encode()
        ).decode()

        if not secrets.compare_digest(password, decrypted_password):

            flash("Invalid email or password.")
            return redirect(url_for("login"))

        # SUCCESSFUL LOGIN

        session.clear()

        session["authenticated"] = True
        session["user_email"] = username

        return redirect(url_for("portfolio"))

    except mysql.connector.Error:

        app.logger.exception("MySQL login error")

        flash("Database connection failed.")
        return redirect(url_for("login"))

    except Exception:

        app.logger.exception("Authentication error")

        flash("Login failed. Please try again.")
        return redirect(url_for("login"))

    finally:

        if cursor is not None:
            cursor.close()

        if db is not None and db.is_connected():
            db.close()



# PORTFOLIO


@app.route("/portfolio")
def portfolio():

    if not session.get("authenticated"):

        flash("Please log in to access your portfolio.")
        return redirect(url_for("login"))

    return redirect(PORTFOLIO_URL)



# LOGOUT

@app.route("/logout")
def logout():

    session.clear()

    flash("You have been logged out.")

    return redirect(url_for("home"))



# RUN FLASK SERVER


print("Flask application is ready.")
print("Home page: http://127.0.0.1:5000/")
print("Stop the server by interrupting the notebook kernel.")

app.run(
    host="127.0.0.1",
    port=5000,
    debug=False,
    use_reloader=False
)

Flask application is ready.
Home page: http://127.0.0.1:5000/
Stop the server by interrupting the notebook kernel.
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [09/Oct/2026 13:33:13] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [09/Oct/2026 13:33:15] "GET /login HTTP/1.1" 200 -
127.0.0.1 - - [09/Oct/2026 13:33:19] "GET /signup HTTP/1.1" 200 -
127.0.0.1 - - [09/Oct/2026 13:33:56] "POST /signup HTTP/1.1" 302 -
127.0.0.1 - - [09/Oct/2026 13:33:56] "GET /signup HTTP/1.1" 200 -
127.0.0.1 - - [09/Oct/2026 13:34:51] "POST /signup HTTP/1.1" 302 -
127.0.0.1 - - [09/Oct/2026 13:34:52] "POST /signup HTTP/1.1" 302 -
127.0.0.1 - - [09/Oct/2026 13:34:52] "GET /verify-otp HTTP/1.1" 200 -
127.0.0.1 - - [09/Oct/2026 13:36:06] "POST /verify-otp HTTP/1.1" 302 -
127.0.0.1 - - [09/Oct/2026 13:36:06] "GET /login HTTP/1.1" 200 -
127.0.0.1 - - [09/Oct/2026 13:36:21] "POST /login HTTP/1.1" 302 -
127.0.0.1 - - [09/Oct/2026 13:36:21] "GET /portfolio HTTP/1.1" 302 -
